# **Glovo Menu Dataset Preparation**

This notebook prepares a Glovo menu dataset for further market analysis.

The goal is to clean, validate, enrich, and structure the data for future analysis of pricing, discounts, menu assortment, and restaurant positioning in the Lviv food delivery market.

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

## **1. Setup & Data Loading**


The dataset contains menu-level information collected from Glovo restaurant pages in Lviv.

Each row represents a single menu item and includes restaurant metadata, menu category, pricing information, discount details, cuisine categories, and product descriptions.

The final dataset was created by combining three data sources:

- Restaurant menu pages (dish-level information)
- Cuisine category pages (restaurant cuisine classification)
- Store metadata extracted from Glovo pages

## **2. Initial Dataset Overview**


In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16121 entries, 0 to 16120
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   restaurant_key       16121 non-null  object 
 1   restaurant_name      16121 non-null  object 
 2   rating               16121 non-null  object 
 3   menu_category        16112 non-null  object 
 4   dish_name            16121 non-null  object 
 5   regular_price        16121 non-null  float64
 6   discount_price       2392 non-null   float64
 7   final_price          16121 non-null  float64
 8   is_discounted        16121 non-null  bool   
 9   discount_pct         2392 non-null   float64
 10  dish_description     14561 non-null  object 
 11  menu_source_file     16121 non-null  object 
 12  menu_parser_type     16121 non-null  object 
 13  cuisine_type         15457 non-null  object 
 14  cuisine_source_file  15457 non-null  object 
 15  store_url            15457 non-null 

In [4]:
print("Dataset shape:", df.shape)

missing_values = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": round(df.isna().mean() * 100, 2)
})

print(missing_values)

print("\nUnique restaurants:", df["restaurant_name"].nunique())
print("Unique restaurant locations:", df["restaurant_key"].nunique())
print("Unique dishes:", df["dish_name"].nunique())

Dataset shape: (16121, 16)
                     missing_count  missing_pct
restaurant_key                   0         0.00
restaurant_name                  0         0.00
rating                           0         0.00
menu_category                    9         0.06
dish_name                        0         0.00
regular_price                    0         0.00
discount_price               13729        85.16
final_price                      0         0.00
is_discounted                    0         0.00
discount_pct                 13729        85.16
dish_description              1560         9.68
menu_source_file                 0         0.00
menu_parser_type                 0         0.00
cuisine_type                   664         4.12
cuisine_source_file            664         4.12
store_url                      664         4.12

Unique restaurants: 226
Unique restaurant locations: 234
Unique dishes: 12144


**Dataset Summary**

- Total rows: 16,121
- Total columns: 16
- Restaurant locations: 234
- Unique restaurant names: 226
- Unique dishes: 12,144

The difference between restaurant locations and restaurant names indicates that some restaurant chains operate multiple locations under the same brand name.

**Data Quality Overview**

The dataset is highly complete:
- No missing restaurant identifiers, restaurant names, dish names, final prices
- Only 9 records are missing menu categories
- Cuisine information is available for over 95% of menu items

Missing values are expected for **discount-related fields** because most products are sold without active discounts.

**Product descriptions** are unavailable for a small subset of menu items (9.68%) because some restaurants do not provide descriptions on their Glovo pages.

**Column Description**

| Column | Description |
|----------|----------|
| restaurant_key | Unique restaurant identifier extracted from the Glovo store URL |
| restaurant_name | Restaurant name displayed on Glovo |
| rating | Restaurant rating shown on Glovo |
| menu_category | Menu section where the dish is listed (e.g., Pizza, Burgers, Desserts) |
| dish_name | Product name |
| regular_price | Original product price |
| discount_price | Discounted product price when available |
| final_price | Final customer price after applying discounts |
| is_discounted | Indicates whether the product is currently discounted |
| discount_pct | Percentage discount calculated from regular and discounted prices |
| dish_description | Product description provided by the restaurant |
| menu_source_file | Original menu HTML file used for parsing |
| menu_parser_type | Parser used to extract the menu (ItemRow or ItemTile layout) |
| cuisine_type | Cuisine category assigned from Glovo cuisine pages |
| cuisine_source_file | Source cuisine category file |
| store_url | Restaurant URL extracted from Glovo |

## **3. Data Quality Assessment**

Before proceeding to the analysis stage, the dataset was validated for potential quality issues, including:

- Duplicate menu items
- Missing values
- Invalid prices
- Rating consistency
- Menu category completeness
- Cuisine category completeness

The objective of this step is to ensure that the dataset accurately represents restaurant menus and does not contain structural issues that could bias the analysis.

### **3.1. Duplicate Records**

In [5]:
# Check duplicate records

duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 3


In [6]:
# Remove exact duplicates

rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before: 16121
Rows after : 16118
Rows removed: 3


**Duplicate Records**

The dataset was checked for fully duplicated records.

Three exact duplicate menu items were identified. All duplicated rows contained identical values across every column and therefore represented duplicate parsing results rather than distinct products.

These records were removed from the dataset.

### **3.2. Price validation**


In [7]:
print("Minimum price:", df["final_price"].min())
print("Maximum price:", df["final_price"].max())

Minimum price: 0.01
Maximum price: 10666.0


In [8]:
df.nsmallest(10, "final_price")[["restaurant_name", "dish_name", "final_price"]]

,restaurant_name,dish_name,final_price
3823,Cinnamon Bakery,"Гарячий шоколад ""Класік Original"" Акція",0.01
466,Nectarine / Нектарин,Сніданок,0.10
3895,2b Fresh. Bowl & breakfast,Створи свій боул,0.10
3896,2b Fresh. Bowl & breakfast,Солодкий конструктор,0.10
5134,NEWGREEN,Фіксатор для навчальних паличок,0.11
1512,Dragon Wok,Збірний боул,1.00
1727,Poke LULU,Збери сам,1.00
5798,Magic Bowls,Збірний боул,1.00
9219,Dragon Wok,Збірний боул,1.00
12377,Момент,Склади свій бокс,1.00


In [9]:
df.nlargest(10,"final_price")[["restaurant_name", "dish_name", "final_price"]]

,restaurant_name,dish_name,final_price
8557,Bar Mushly,SEAFOOD BOX великий 4860г,10666.0
8558,Bar Mushly,SEAFOOD BOX середній 3430г,7066.0
8575,Bar Mushly,Лобстер Термідор,5866.0
8576,Bar Mushly,Лобстер Chix з соусом Айолі та овочевою сальсою,5602.0
8559,Bar Mushly,SEAFOOD BOX малий 2230г,4666.0
8560,Bar Mushly,Велике плато морепродуктів (750г),4666.0
8513,ChaChito,Велика тарілка гриль (2250г),3950.0
8577,Bar Mushly,Велике плато гриль (800г),3826.0
5316,Babo Gardens / Бабо Гарденс,Велика тарілка грильованих морепродуктів і риб...,3800.0
15559,Roll Club,Найбільший сет !,3243.8


**Price Validation**

Menu prices were reviewed to identify potential data-entry errors and unrealistic values.

The minimum observed price was 0.01 UAH, while the maximum price reached 10,666 UAH.

Manual inspection showed that low-price observations were mainly service-related items, add-ons, or promotional placeholders, while high-price observations corresponded to premium sets and large catering products.

No obvious pricing anomalies requiring removal were detected.

### **3.3 Cuisine Coverage**

In [10]:
# Calculate the share of menu items with available cuisine information

cuisine_coverage = round(100 * df["cuisine_type"].notna().mean(), 2)

print(f"Rows with cuisine information: {cuisine_coverage}%")

Rows with cuisine information: 95.88%


In [11]:
df[df["cuisine_type"].isna()]["restaurant_name"].nunique()

11

Cuisine information was available for the majority of menu items.

A small number of restaurants were not present in the collected cuisine category pages and therefore remained without cuisine labels.

## **4. Data Cleaning & Feature Engineering**

This section creates cleaned and analysis-ready features while preserving the original raw columns.

### **4.1. Rating Cleaning**

Restaurant ratings were originally stored as text values and contained three different formats:

- Percentage ratings (e.g. 97%)
- "Новинка" (new restaurant)
- "--" (rating unavailable)

To enable numerical analysis, percentage ratings were converted into numeric values, while special labels were handled separately.

In [12]:
# Count restaurants without a rating and restaurants marked as new

print("Restaurants without rating:", (df["rating"] == "--").sum())
print("New restaurants:", (df["rating"] == "Новинка").sum())

Restaurants without rating: 2279
New restaurants: 149


In [13]:
# Remove the percentage sign from rating values
df["rating_clean"] = (df["rating"].str.replace("%", "", regex=False))

# Convert percentage ratings to numeric values
# Special values such as "--" and "Новинка" become NaN
df["rating_clean"] = pd.to_numeric(df["rating_clean"],errors="coerce")

# Create a binary flag for restaurants marked as new
df["is_new_restaurant"] = (df["rating"] == "Новинка")

In [14]:
# Review the distribution of cleaned numerical ratings

df["rating_clean"].describe()

,rating_clean
count,13690.000000
mean,97.191015
std,3.038789
min,82.000000
25%,96.000000
50%,98.000000
75%,100.000000
max,100.000000


Most restaurants demonstrated very high customer ratings, with a median rating of 98%.

Restaurants marked as "Новинка" were preserved using a separate binary feature, while missing ratings remained as null values.

### **4.2. Menu Category Standardization**

One of the main challenges of the dataset was the large number of menu category names.

Although restaurants often sell similar products, they use highly inconsistent category labels. For example:

- "ПІЦА"
- "ПІЦА 30СМ"
- "ПІЦА BBQ"
- "НЕАПОЛІТАНСЬКА ПІЦА"

all represent the same business concept.

In [15]:
# Standardize menu category text

df["menu_category_clean"] = (df["menu_category"].str.strip().str.upper())

print("Raw menu categories:", df["menu_category"].nunique())
print("Cleaned menu categories:", df["menu_category_clean"].nunique())

Raw menu categories: 1357
Cleaned menu categories: 1242


After basic text normalization, the number of unique menu categories decreased from 1,357 to 1,242, which still remained too fragmented for meaningful analysis.

In [16]:
# Review most common cleaned menu categories

category_counts = (df["menu_category_clean"].value_counts().reset_index())

category_counts.columns = ["menu_category_clean", "items_count"]

display(category_counts.head(10))

,menu_category_clean,items_count
0,НАПОЇ,1074
1,РОЛИ,695
2,САЛАТИ,604
3,ХІТ ПРОДАЖІВ,516
4,ДЕСЕРТИ,479
5,СОУСИ,461
6,ХОЛОДНІ НАПОЇ,454
7,СНІДАНКИ,347
8,ПІЦА,338
9,ОСНОВНІ СТРАВИ,296


In [17]:
# Review suspicious categories by parser type

df.groupby("menu_parser_type")["menu_category_clean"].nunique().sort_values(ascending=False)

,menu_category_clean
menu_parser_type,
ItemRow,769
ItemTile,492


In [18]:
# Inspect rare categories

rare_categories = (df["menu_category_clean"].value_counts())

rare_categories = rare_categories[rare_categories <= 10].index

df[
    df["menu_category_clean"].isin(rare_categories)
][
    [
        "restaurant_name",
        "menu_category_clean",
        "dish_name",
        "menu_parser_type"
    ]
].sample(10)

,restaurant_name,menu_category_clean,dish_name,menu_parser_type
9779,MVF / МВФ,ПРОХОЛОДНІ НАПОЇ,Фанта (250),ItemRow
251,Daily Dose,ВЛАСНА ПРОДУКЦІЯ,Котлети з індички 4 шт.,ItemRow
11427,Vasylevi pyrohy/Пекарня-кав'ярня Василеві пироги,ПЕЛЬМЕНІ З ІНДИКА С/М (600Г),"Вареник чорниця , с/м 600г",ItemTile
12329,Паб Добрий Друг,СТРАВИ НА ВОГНІ,Овочі гриль з соусом Чімічурі,ItemRow
7748,Момент на УПА,SOFT ДРІНКИ,Напій Sprite (250мл),ItemRow
6634,OKKO CAFÉ,КАРТОПЛЯ ФРІ (МАЛЕНЬКА) M,Картопля фрі (стандартна) L,ItemTile
16062,SNACK&BURGERS,БРІОШІ,Бріош Дабл Чікен Де Люкс (300г),ItemRow
14590,Шаурма 5 зірок,ШАУРМА ГРУЗИНСЬКА,Шаурма грузинська мікс велика (700 гр),ItemRow
8955,Maioran / Майоран,МЛИНЦІ/КІШІ,Млинці з сиром (300/50г),ItemRow
6735,OKKO CAFÉ,"НАПІЙ ФАНТА АПЕЛЬСИН СИЛ/ГАЗ 0,5Л","Нектар Садочок Виноград-ябл зел 0,2л",ItemTile


In [19]:
df[df["menu_parser_type"] == "ItemTile"]["restaurant_name"].value_counts()

,count
restaurant_name,
OKKO CAFÉ,279
BISCOTTI,69
Vasylevi pyrohy/Пекарня-кав'ярня Василеві пироги,60
Сімейна Пекарня,54
Cinnamon Bakery,36
Кооператива Молока. Milk Cafe,36


#### **Correcting ItemTile Categories**



During category validation, some suspicious category values were identified, including individual product names and categories appearing only once.

Further inspection showed that many of these cases were related to records parsed with the ItemTile layout.

Unlike the standard ItemRow layout, some ItemTile pages stored product names instead of actual menu sections. To improve category consistency, ItemTile records were reclassified using dish-name based rules.

In [20]:
# Correct unreliable ItemTile categories using dish-name based rules

def assign_itemtile_category(row: pd.Series) -> str:
    """
    Recover menu categories for records parsed with the ItemTile parser.

    The ItemTile layout occasionally stores product names instead of
    actual menu section names. This function uses dish-name keywords
    to assign a more reliable category before category standardization.

    Returns:
        Cleaned menu category.
    """

    name = str(row["dish_name"]).upper()

    if any(x in name for x in ["КОМБО", "МЕНЮ"]):
        return "КОМБО"

    if any(x in name for x in ["ЛАТЕ", "АМЕРИКАНО", "КАПУЧ", "ЕСПРЕСО", "РАФ",
                               "КАВА", "ФЛЕТ"]):
        return "КАВА"

    if any(x in name for x in ["СІК", "ЧАЙ", "КАКАО", "НАПІЙ", "ВОДА", "СОДОВА",
                               "ЛИМОНАД", "COCA", "КОКА", "PEPSI", "BURN", "RED BULL",
                               "ФАНТА", "СПРАЙТ", "СМУЗІ", "7UP"]):
        return "НАПОЇ"

    if any(x in name for x in ["САЛАТ", "ЦЕЗАР"]):
        return "САЛАТИ"

    if any(x in name for x in ["КАРТОПЛ", "ФРІ", "НАГЕТСИ"]):
        return "ФРІ"

    if "ПІЦА" in name or "КАЛЬЦОНЕ" in name:
        return "ПІЦА"

    if "БУРГЕР" in name:
        return "БУРГЕРИ"

    if "СЕНДВІЧ" in name:
        return "СЕНДВІЧІ"

    if "СНІДАН" in name or "СИРНИКИ" in name:
        return "СНІДАНОК"

    if "ХОТ-ДОГ" in name or "ХОТ ДОГ" in name:
        return "ХОТ-ДОГИ"

    if "СОУС" in name:
        return "СОУС"

    if any(x in name for x in ["КРУАСАН", "БУЛОЧКА", "ПИРІГ", "PIE", "ПИРІЖОК", "ХЛІБ",
                               "ФОКАЧА", "ПАМПУШК", "СЛОЙКА", "БУЛКА", "ХАЧАПУР"]):
        return "ВИПІЧКА"

    if any(x in name for x in ["ТІРАМІСУ", "ТИРАМІСУ", "МЕДІВНИК", "ТОРТ", "ІРИС",
                               "СИРОК", "ТРУБОЧКА", "КАРТОПЛИНКА", "ДЕСЕРТ", "МАКАРУН",
                               "ЗЕФІР", "СИРНИК", "ЧІЗКЕЙК", "МАФІН", "ПЕЧИВО", "СОЛОДК",
                               "ШОКОЛАД", "ФІСТАШК", "КАРАМЕЛЬ", "МАНГО-МАРАК",
                               "ВИШН", "МАЛИН", "КРЕМ", "КОРИЦ", "СИРОП", "ПОЛУНИ",
                               "ГОРІХ", "МАК", "МИГДАЛ", "ТІСТЕЧКО", "ЕКЛЕР", "ДОНАТ",
                               "МАФІН", "MACARON", "ПАНАКОТ"]):
        return "ДЕСЕРТИ"

    return "ІНШЕ"

In [21]:
# Apply ItemTile correction

itemtile_mask = df["menu_parser_type"] == "ItemTile"

df.loc[itemtile_mask, "menu_category_clean"] = (
    df.loc[itemtile_mask]
      .apply(assign_itemtile_category, axis=1)
)

print("Menu categories after ItemTile correction:", df["menu_category_clean"].nunique())

print("Remaining ItemTile categories classified as 'ІНШЕ':", (
    (df["menu_parser_type"] == "ItemTile") & (df["menu_category_clean"] == "ІНШЕ")
    ).sum()
)

Menu categories after ItemTile correction: 770
Remaining ItemTile categories classified as 'ІНШЕ': 104


Some ItemTile records were assigned to ІНШЕ. These were later reviewed during analytical grouping and mostly represented non-food or retail products.

#### **Analytical Menu Category Grouping**

After basic cleaning, the dataset still contained 770 menu category variations.

To make categories comparable across restaurants, a rule-based grouping function was created.

The logic combines:

- business categories such as Bestsellers, Promotions, New Items;
- broad food groups such as Pizza, Sushi & Rolls, Burgers, Desserts;
- manually reviewed exceptions for ambiguous categories;
- keyword-based rules for scalable classification.

The result is stored in a new column: menu_category_group.

In [22]:
# Group cleaned menu categories into broader analytical categories

def assign_menu_category_group(category: str) -> str:
    """
    Assign a standardized menu category group.

    This function maps raw menu categories into a smaller set of
    analytical groups that can be used for market-level analysis,
    category comparisons, pricing studies, and assortment evaluation.

    Categories that do not match any predefined rule are assigned
    to the "Other" group for further review.

    Parameters
    category : str
        Original menu category name.

    Returns
    str
        Standardized menu category group.
    """
    cat = str(category).upper().strip()

    if "ХІТ ПРОДАЖІВ" in cat:
        return "Bestsellers"

    if "ПРОМО" in cat or "WOW PRICE" in cat:
        return "Promotions"

    if any(x in cat for x in ["СТРАВА МІСЯЦЯ", "JUNGLE BAR", "SPECIAL",
                              "(НЕ)СВЯТА ГАЛИЧИНА", "ПОГРАБУВАННЯ", "СПЕШ",
                              "СЕЗОННІ", "ПОЛУНИ", "ВЕСНЯН", "POP-UP"]):
        return "Special"

    if any(x in cat for x in ["ГОТУЄМО", "ГОТУЙ", "ЗАМОРО", "SHOP", "ВЛАСНА ПРОДУКЦІЯ",
                              "ЗАПАСИ", "БЛАГОДІЙНІСТЬ", "ПАКЕТ", "ПРИБОРИ",
                              "ІНШІ ТОВАРИ", "БРЕНДОВАНІ ТОВАРИ", "ДРІПИ", "ЛІНІЙКА KREDENS",
                              "ЩОСЬ ДОБРЕ", "IНШЕ", "ІНШЕ"]):
        return "Goods to go"

    if "НОВИНК" in cat or "НОВІ" in cat:
        return "New Items"

    if any(x in cat for x in ["ОБІД", "ЛАНЧ", "БІЗНЕС-ЛАНЧ", "КОМПЛЕКСН"]):
        return "Business Lunches"

    if any(x in cat for x in ["КОМБО", "КOМБО", "СЕТ", "НАБОР", "БОКС",
                              "BOX", "МЕНЮ", "КОМПАНІ", "МІКС",
                              "ВІД МІС ЛІ", "ПРОПОЗИЦІЯ ТИЖНЯ"]):
        return "Sets & Combos"

    if any(x in cat for x in ["ПІЦ", "PIZZA", "ПІНСА", "ПАНУОЦУ"]):
        return "Pizza"

    if any(x in cat for x in ["БУРГЕР", "ЧІЗБУРГЕР", "ГАМБУРГЕР"]):
        return "Burgers"

    if "ХОТ-ДОГ" in cat or "ХОТ ДОГ" in cat:
        return "Hot Dogs"

    if "ХІНКАЛ" in cat:
        return "Khinkali"

    if "ХАЧАПУР" in cat or "ПІДЕ" in cat:
        return "Khachapuri"

    if any(x in cat for x in ["ЧЕБУРЕК", "ЯНТИК", "TUNA MENU"]):
        return "Chebureki & Yantyky"

    if any(x in cat for x in ["РИБ", "МОРЕПРОДУК", "КРЕВЕТ", "SEAFOOD", "FISH", "ЇСТИ РУКАМИ"]):
        return "Fish & Seafood"

    if any(x in cat for x in ["КУРКА", "КУРЯЧ", "ПТИЦЯ", "КРОЛИК", "КАЧК", "ІНДИЧ", "КРИЛАТІ"]):
        return "Poultry & Rabbit"

    if any(x in cat for x in ["М'ЯС", "МʼЯС", "СВИНИНА", "ЯЛОВИЧИНА", "ТЕЛЯТИНА", "БАРАНИНА",
                              "ЯГНЯ", "РЕБРА", "КОВБАС"]):
        return "Meat Dishes"

    if any(x in cat for x in ["ГРИЛЬ", "МАНГАЛ", "BBQ", "ХОСПЕР", "ВОГН", "СТЕЙКИ"]):
        return "Grill"

    if any(x in cat for x in ["ПАСТ", "РИЗОТ", "РАВІОЛ", "НЬОКІ", "ЛАЗАНЬЯ", "РІЗОТО", "PASTA"]):
        return "Pasta & Risotto"

    if any(x in cat for x in ["WOK", "ВОК", "КАРІ"]):
        return "WOK"

    if any(x in cat for x in ["БОУЛ", "BOWL", "ПОКЕ", "ЗБЕРИ САМ"]):
        return "Bowls"

    if any(x in cat for x in ["СУП", "ЗУП", "РАМЕН", "БУЛЬЙОН", "ЧОРБ", "ТОМ ЯМ", "ПЕРШІ СТРАВИ"]):
        return "Soups"

    if "САЛАТ" in cat or "РІЗНА ЗЕЛЕНЬ" in cat:
        return "Salads"

    if any(x in cat for x in ["ХОЛОДНІ ЗАКУСКИ", "ЗАКУСКИ ХОЛОДНІ", "БРУСКЕТ",
                              "АНТІПАСТ", "АНТИПАСТ", "ХУМУС", "МЕЗЕ", "ЧИПСИ",
                              "ГОРІХИ", "НАЧОС", "КАВ'ЯР БАР", "УСТРИЧНИЙ БАР",
                              "ДО ЗАСТІЛЛЯ"]):
        return "Cold Appetizers"

    if any(x in cat for x in ["ЗАКУСК", "СНЕК", "СТАРТЕР", "ФРИТЮР", "ФРІ", "FRIES",
                              "КАРТОПЛЯ", "НАГЕТС", "КРИЛЬЦ", "ДО ПИВА", "ЦИБУЛЕВІ КІЛЬЦЯ",
                              "СИР ФРІ", "ФАЛАФЕЛЬ", "БАТАТ", "САЙДИ", "КУМПІР",
                              "ВЕДЖІ", "БАКЕТИ", "ПАЛИЧКИ МОЦАРЕЛА", "ДОПИ"]):
        return "Hot Appetizers & Fried Snacks"

    if any(x in cat for x in ["ГАРНІР", "САЙД", "ОВОЧІ ТА КАШІ", "ДО РИБИТА МОРЕПРОДУКТІВ", "А ЩЕ"]):
        return "Side Dishes"

    if any(x in cat for x in ["СОУС", "ДОДАТК", "ДОПОВНЕННЯ", "ДІП", "ДО СТРАВ", "ФЕРМЕНТАЦІЯ"]):
        return "Sauces & Add-ons"

    if any(x in cat for x in ["СНІДАН", "БЕНЕДИКТ", "БРАНЧ", "ВІВСЯНКА", "СИРНИК",
                              "СИРНИЧК", "ГОФР", "MORNING"]):
        return "Breakfast"

    if any(x in cat for x in ["СЕНДВІЧ", "СЕНДВIЧ", "ТОСТ", "МЕЛТ", "ПАНІНІ",
                              "БЕЙГЛ", "ЧІАБАТА", "БУТЕРБРОД"]):
        return "Sandwiches & Melts"

    if "ПАНКЕЙК" in cat:
        return "Pancakes"

    if any(x in cat for x in ["МЛИНЦ", "НАЛИСНИК", "GLOVO/BOLT"]):
        return "Crepes"

    if any(x in cat for x in ["ВАРЕНИК", "ПЕЛЬМЕН", "ВУШКА"]):
        return "Varenyky & Pelmeni"

    if "ДЕРУН" in cat:
        return "Deruny"

    if any(x in cat for x in ["ГЬОЗА", "ГЬОДЗА", "ДАМПЛІНГИ"]):
        return "Gyoza & Dumplings"

    if any(x in cat for x in ["БУРІТО", "ВРАП", "КЕСАДИЛ", "ТАКО", "FIESTA", "МЕКСИКИ"]):
        return "Burritos, Wraps & Tacos"

    if any(x in cat for x in ["ЛАФА", "ЛАФИ", "BATBUT", "ГІРОСИ", "ВЕЛИКА КИШЕНЯ"]):
        return "Lafa, Giros & Pita"

    if any(x in cat for x in ["ДЕСЕРТ", "СОЛОД", "ТОРТ", "ЧІЗКЕЙК", "ЧИЗКЕЙК",
                              "МАКАРУН", "МАКАРОНИ", "ПЕЧИВО", "ЦУКЕРК", "ЕКЛЕР",
                              "ДОНАТ", "МОРОЗИВО", "ПЛЯЦК", "ШОКОЛАД", "МАФІН",
                              "ДО КАВИ", "БЕЗ ЦУКРУ", "ЛЬОДЯНИКИ", "КАРАМЕЛЬ", "ДРАЖЕ",
                              "ЦІКАВИНКИ",	"ДЛЯ ДОРОСЛИХ", "EMILY & DREAMERS",
                              "ФІГУРКИ"]):
        return "Desserts"

    if any(x in cat for x in ["ДЛЯ НАЙМЕНШИХ", "ДИТЯЧЕ", "KIDS", "ХЕППІ МІЛ", "МАЛЕЧІ"]):
        return "Kids Menu"

    if any(x in cat for x in ["ОСНОВН", "ВЕСЬ ДЕНЬ", "ДРУГІ", "ГАРЯЧІ СТРАВИ",
                              "ГАРЯЧІ", "ВЕГГІ", "ТАРІЛК", "СТРАВИ", "МЕЙНИ",
                              "ГОЛОВНЕ", "РИС", "ЛОКШИНА", "ЗАПІКАНК", "ЇЖА",
                              "СИТНО", "КАРТОПЛЯ НА СКОВОРОДІ", "РАВЛИКИ", "ЩОСЬ БІЛЬШЕ",
                              "ПОТАТА", "АЗІЯ"]):
        return "Main Dishes"

    if any(x in cat for x in ["КЕБАБ", "КЕБAБ", "ШАУРМА", "ДОНЕР", "DÖNER", "ЛАВАШ",
                              "КЛАСИЧНИЙ", "АВТОРСЬКИЙ", "СИРНИЙ ЧІЗІ-БУМ", "ОВОЧЕВИЙ",
                              "ДАБЛ МІТ"]):
        return "Kebabs & Shawarma"

    if any(x in cat for x in ["РОЛ", "СУШІ", "МАКІ", "ФІЛАДЕЛЬФ", "КАЛІФОРН", "САШИМІ", "ROLLS",
                              "ДРАКОН", "ТЕМПУР", "НОРІ", "ФУТО", "ФІРМОВІ", "ГРІН", "БЛЕК",
                              "ПРЕМІУМ", "ЗАПЕЧЕНІ", "АВТОРСЬКІ", "ВЕРШКОВІ", "ТЕПЛІ", "СИРНІ",
                              "СМАЖЕНІ"]):
        return "Sushi & Rolls"

    if any(x in cat for x in ["ВИПІЧКА", "ХЛІБ", "КРУАСАН", "БУЛОЧКА", "ФОКАЧА", "ПИРІГ",
                              "ПИРІЖ", "ПАМПУШК", "БРІОШ", "ПИРОГИ", "ТІСТО", "СЛОЙКИ",
                              "HAISTER Х MITTE", "БАО", "СМАЖЕНЕ"]):
        return "Bakery"

    if any(x in cat for x in ["НАПО", "НАПІЙ", "ПЛЯШКИ", "КАВА", "ЧАЙ", "ЛИМОНАД",
                              "СМУЗІ", "ФРЕШ", "ВОДА", "СІК", "СОКИ", "КОМБУЧА",
                              "МАТЧА", "BUBBLE", "БАБЛ", "МІЛКШЕЙК", "КОКТЕЙЛ", "DRINK",
                              "SOFT", "СОФТ", "ПЕВО", "ПИВО", "КРАФТИ", "ШОТИ", "УЗВАР",
                              "КОМПОТ", "МОРС", "CLUB MATE", "АЙРАН", "ЛІКЕРИ", "БЕЗАЛКО",
                              "ХОУММЕЙД", "САНГРІТА", "БАР"]):
        return "Drinks"

    if "ДЛЯ ХВОСТИКІВ" in cat:
        return "Pet Food / Exclude"

    return "Other"

In [23]:
# Apply analytical categories

df["menu_category_group"] = df["menu_category_clean"].apply(assign_menu_category_group)

print(df["menu_category_group"].value_counts())

menu_category_group
Drinks                           2656
Desserts                         1183
Sushi & Rolls                    1176
Pizza                            1015
Sets & Combos                    1013
Main Dishes                       953
Sauces & Add-ons                  951
Hot Appetizers & Fried Snacks     895
Salads                            707
Bestsellers                       516
Breakfast                         487
Soups                             448
Kebabs & Shawarma                 403
Burgers                           363
Bakery                            314
Pasta & Risotto                   309
Promotions                        248
Cold Appetizers                   221
Side Dishes                       221
Goods to go                       218
Sandwiches & Melts                198
Meat Dishes                       153
WOK                               150
Fish & Seafood                    143
Varenyky & Pelmeni                126
Bowls                         

**Category Coverage Check**

After applying the rule-based grouping function, all menu categories were successfully assigned to an analytical group.

The share of uncategorized records was checked using the Other category.

In [24]:
other_share = (df["menu_category_group"] == "Other").mean() * 100

print(f"Other share: {other_share:.2f}%")

Other share: 0.00%


#### **Non-Food Product Removal**

Some Glovo restaurant pages also contained retail or non-menu products, such as packaged goods, frozen products, coffee products, branded items, and pet food.

Since the goal of this project is to analyze restaurant food menus, these non-food product groups were excluded from further analysis.

In [25]:
goods_to_go_check = df[
    df["menu_category_group"] == "Goods to go"
][
    ["menu_category_clean", "restaurant_name", "dish_name", "final_price"]
].sort_values(
    ["menu_category_clean", "restaurant_name"]
)

display(goods_to_go_check)

,menu_category_clean,restaurant_name,dish_name,final_price
6130,IНШЕ,KFC,ЕКО-ШОПЕР,29.0
6131,IНШЕ,KFC,ПАКЕТ,6.0
11092,KITCHEN SHOP,Cukor RED/Цукор Red,Беконовий джем,339.0
11093,KITCHEN SHOP,Cukor RED/Цукор Red,Мигдалевий крем,247.0
11094,KITCHEN SHOP,Cukor RED/Цукор Red,Кімчі,207.0
...,...,...,...,...
4259,СТОЛОВІ ПРИБОРИ ТА ДОДАТКИ,МУЗА від Євгена Клопотенка / MUZA vid Yevhena ...,"Набір еко приборів (виделка, ніж та 2 серветки)",5.0
4260,СТОЛОВІ ПРИБОРИ ТА ДОДАТКИ,МУЗА від Євгена Клопотенка / MUZA vid Yevhena ...,Ложка супова одноразова еко,2.0
6434,ЩОСЬ ДОБРЕ,Code/Код,Зерно Mad Heads (250g),590.0
6435,ЩОСЬ ДОБРЕ,Code/Код,Кавові дріпи Mad Heads/ LOUMI,590.0


In [26]:
# Remove non-food and non-menu product categories

rows_before = len(df)

df = df[~df["menu_category_group"].isin(["Goods to go", "Pet Food / Exclude"])]

rows_after = len(df)

print("Rows removed:", rows_before - rows_after)
print("Rows remaining:", rows_after)

Rows removed: 220
Rows remaining: 15898


#### **Final Menu Category Refinement**

Several small but conceptually similar categories were merged to improve interpretability and reduce fragmentation.

The following groups were consolidated:

- Khachapuri and Khinkali - Georgian Cuisine
- WOK and Gyoza & Dumplings - Asian Dishes
- Kebabs & Shawarma and Lafa/Giros/Pita - Kebabs, Shawarma & Gyros
- Crepes and Pancakes - Crepes & Pancakes

In [27]:
print(df["menu_category_group"].value_counts(normalize=True) * 100)

menu_category_group
Drinks                           16.706504
Desserts                          7.441188
Sushi & Rolls                     7.397157
Pizza                             6.384451
Sets & Combos                     6.371871
Main Dishes                       5.994465
Sauces & Add-ons                  5.981885
Hot Appetizers & Fried Snacks     5.629639
Salads                            4.447100
Bestsellers                       3.245691
Breakfast                         3.063278
Soups                             2.817965
Kebabs & Shawarma                 2.534910
Burgers                           2.283306
Bakery                            1.975091
Pasta & Risotto                   1.943641
Promotions                        1.559945
Side Dishes                       1.390112
Cold Appetizers                   1.390112
Sandwiches & Melts                1.245440
Meat Dishes                       0.962385
WOK                               0.943515
Fish & Seafood                    

In [28]:
# Merge closely related categories

category_merge_map = {
    "Khachapuri": "Georgian Cuisine",
    "Khinkali": "Georgian Cuisine",
    "WOK": "Asian Dishes",
    "Gyoza & Dumplings": "Asian Dishes",
    "Kebabs & Shawarma": "Kebabs, Shawarma & Gyros",
    "Lafa, Giros & Pita": "Kebabs, Shawarma & Gyros",
    "Pancakes": "Crepes & Pancakes",
    "Crepes": "Crepes & Pancakes"
}

df["menu_category_group"] = (df["menu_category_group"].replace(category_merge_map))

In [29]:
print("Unique categories before grouping:", df["menu_category_clean"].nunique())
print("Unique categories after grouping:", df["menu_category_group"].nunique())

reduction_pct = (
    1 - df["menu_category_group"].nunique()
    / df["menu_category_clean"].nunique()
) * 100

print(f"Category reduction: {reduction_pct:.1f}%")

Unique categories before grouping: 750
Unique categories after grouping: 37
Category reduction: 95.1%


**Category Grouping Results**

After basic text normalization, the dataset contained **1,242 cleaned category names**.

After correcting ItemTile categories, this number decreased to **750**.

Finally, rule-based grouping consolidated these 750 category names into **37 analytical category groups**, reducing category fragmentation by **95.1%** while preserving meaningful distinctions between menu sections such as Pizza, Sushi & Rolls, Main Dishes, Drinks, Desserts, and Breakfast.

The resulting category structure provides a consistent foundation for further analysis of menu assortment, pricing, discounts, and restaurant positioning across the Lviv food delivery market.

### **4.3. Cuisine Type Standardization**


#### **Cuisine Type Completion**



Cuisine information was missing for a small number of restaurants because these restaurants were not present in the collected cuisine-category pages.

After reviewing restaurant names and menu offerings, cuisine labels were manually assigned to 11 restaurants.

This step increased cuisine coverage from 95.8% to 100%.

In [30]:
# Identify restaurants with missing cuisine labels after category-page integration

missing_cuisine = (
    df[df["cuisine_type"].isna()]
    [["restaurant_name", "restaurant_key"]]
    .drop_duplicates()
    .sort_values("restaurant_name")
)

print("Restaurants without cuisine:", len(missing_cuisine))

print(missing_cuisine)

Restaurants without cuisine: 11
                restaurant_name               restaurant_key
8715         Coffee Lab Гнатюка      coffee-lab-gnatyuka-lvi
7751          Coffee Lab Франка        coffee-lab-franka-lvi
11869   Coffee lab Трускавецька               coffee-lab-lvi
2390   Fast Fish / Швидка Рибка  fast-fish-shvidka-ribka-lvi
12656                      Lume                 delmarko-lvi
1594              Shisha Garden            shisha-garden-lvi
493                       Tutti              tutti-lvi-1n7je
695                        koji                     koji-lvi
2053                    Андижан                 andizhan-lvi
9018              Дерун / Derun                    derun-lvi
12374                    Момент                   moment-lvi


In [31]:
# Manually assign cuisine labels for restaurants missing from cuisine category pages

manual_cuisine_fill = {
    "Coffee Lab Гнатюка": "Солодощі, Сніданок",
    "Coffee Lab Франка": "Солодощі, Сніданок",
    "Coffee lab Трускавецька": "Солодощі, Сніданок",
    "Fast Fish / Швидка Рибка": "Морепродукти, Фаст-фуд",
    "Lume": "Бургери, Фаст-фуд",
    "Shisha Garden": "Бургери, Фаст-фуд",
    "Tutti": "Італійська, Піца",
    "koji": "Азіатська",
    "Андижан": "Гриль",
    "Дерун / Derun": "Українська, Сніданок",
    "Момент": "Солодощі, Європейська"
}

In [32]:
# Fill missing cuisine values using restaurant names

df["cuisine_type"] = df["cuisine_type"].fillna(
    df["restaurant_name"].map(manual_cuisine_fill)
)

In [33]:
# Confirm that all cuisine values were completed

print("Remaining missing cuisines:", df["cuisine_type"].isna().sum())

Remaining missing cuisines: 0


Missing cuisine labels were manually assigned only for 11 restaurants (4.8% of restaurant brands), making the impact on the overall dataset negligible.

#### **Cuisine Type Translation**

Cuisine categories were extracted from Glovo cuisine category pages.

Since the original cuisine labels were stored in Ukrainian, they were translated into standardized English labels to make the final dataset easier to use in further analysis and dashboarding.

Restaurants may belong to multiple cuisine categories, so combined cuisine labels were preserved instead of forcing each restaurant into a single cuisine group.

In [34]:
# Review the number of unique cuisine combinations after completion

print("Unique cuisine types:", df["cuisine_type"].nunique())

Unique cuisine types: 92


In [35]:
# Extract all base cuisine labels from comma-separated cuisine combinations

base_cuisines = sorted(
    set(
        part.strip()
        for cuisines in df["cuisine_type"].dropna()
        for part in cuisines.split(",")
    )
)

base_cuisines

['Європейська',
 'Європейська',
 'Італійська',
 'Італійська',
 'Азіатська',
 'Американська',
 'Арабська',
 'Бургери',
 'Вегетаріанська',
 'Гриль',
 'Грузинська',
 'Десерти',
 'Корисна їжа',
 'Морепродукти',
 'Міжнародна',
 'Пекарня',
 'Піца',
 'Сніданок',
 'Солодощі',
 'Суші',
 'Українська',
 'Українська',
 'Фаст-фуд',
 'Чай і кава',
 'Шаурма']

In [36]:
# Translation dictionary for base cuisine labels

cuisine_translation = {
    "Європейська": "European",
    "Італійська": "Italian",
    "Азіатська": "Asian",
    "Американська": "American",
    "Арабська": "Arabic",
    "Бургери": "Burgers",
    "Вегетаріанська": "Vegetarian",
    "Гриль": "Grill",
    "Грузинська": "Georgian",
    "Десерти": "Desserts & Coffee",
    "Корисна їжа": "Healthy Food",
    "Морепродукти": "Seafood",
    "Міжнародна": "International",
    "Пекарня": "Bakery",
    "Піца": "Pizza",
    "Сніданок": "Breakfast",
    "Солодощі": "Desserts & Coffee",
    "Суші": "Sushi",
    "Українська": "Ukrainian",
    "Фаст-фуд": "Fast Food",
    "Чай і кава": "Desserts & Coffee",
    "Шаурма": "Kebab & Shawarma",
}

In [37]:
def normalize_text(text):
    """
    Normalize text values before translation.

    This helps handle hidden spaces and different Unicode forms
    that may visually look identical but are treated differently by Python.
    """
    if pd.isna(text):
        return np.nan

    return (
        unicodedata.normalize("NFC", str(text))
        .replace("\u00a0", " ")
        .replace("’", "'")
        .replace("`", "'")
        .strip()
    )

In [38]:
# Normalize dictionary keys before applying translation

cuisine_translation_normalized = {
    normalize_text(key): value
    for key, value in cuisine_translation.items()
}

In [39]:
def translate_cuisine(cuisine):
    """
    Translate Glovo cuisine categories into standardized English labels.

    Multiple cuisine labels assigned to one restaurant are preserved.
    """
    if pd.isna(cuisine):
        return np.nan

    cuisines = [
        normalize_text(x)
        for x in str(cuisine).split(",")
    ]

    translated = [
        cuisine_translation_normalized.get(x, x)
        for x in cuisines
    ]

    return ", ".join(translated)

In [40]:
df["cuisine_type_en"] = df["cuisine_type"].apply(translate_cuisine)

In [41]:
# Validate that every base cuisine label is covered by the translation dictionary

base_cuisines = sorted(
    set(
        normalize_text(part)
        for cuisines in df["cuisine_type"].dropna()
        for part in cuisines.split(",")
    )
)

missing_in_translation = [
    cuisine
    for cuisine in base_cuisines
    if cuisine not in cuisine_translation_normalized
]

print("Missing in translation dictionary:", missing_in_translation)

Missing in translation dictionary: []


### **4.4. Restaurant Name Standardization**





Restaurant names were preserved in their original form and additionally standardized into a separate column.

The standardized version is useful for future matching, dashboard labels, and analysis where consistent Latin naming is preferred.

In [42]:
df["restaurant_name_original"] = df["restaurant_name"]

In [43]:
# Preview restaurants with bilingual names
df.loc[
    df["restaurant_name"].str.contains("/", regex=False, na=False),
    "restaurant_name"
].unique()

array(['Dovhi Burkhlyvi Oplesky / Довгі Бурхливі Оплески',
       "PAB Miaso / ПАБ М'ясо", 'Nectarine / Нектарин', 'TALI / ТАЛІ',
       'Biliardnyi klub «Be to Billiard» / Більярдний клуб «Be to Billiard»',
       'Pizza Hot / Піца Хот', 'MegaBulka / МегаБулка',
       'Fast Fish / Швидка Рибка', 'Cukor Black/ Цукор Black',
       'МУЗА від Євгена Клопотенка / MUZA vid Yevhena Klopotenka',
       'Mamyni Deruny / Мамині Деруни', 'Панда Суші / Panda Sushi',
       'Львівська майстерня шоколаду / Lvivska maisternia shokoladu',
       'Kebabtsia / Кебабця', "Бургери LA П'ЄЦ / Burgers LA PIEC",
       'Babo Gardens / Бабо Гарденс', 'Royal Kebab / Роял Кебаб',
       'Home restaurant / Домашній ресторанчик', 'Ya Hrek / Я Грек',
       'Code/Код', "Kebab LA PIEC / Кебаб LA П'ЄЦ",
       'Zashkvarka / Зашкварка', 'Burek / Бурек',
       'Chornomorka / Чорноморка', 'Persha Zapikanka / Перша Запіканка',
       'Kebab super / Кебаб Супер', 'Bilyi Shum / Білий Шум',
       'INTEMPO / ІНТЕМПО', 

In [44]:
# Manually standardize known bilingual restaurant brands
name_replacements = {
    "МУЗА від Євгена Клопотенка / MUZA vid Yevhena Klopotenka": "MUZA vid Klopotenka",
    "Панда Суші / Panda Sushi": "Panda Sushi",
    "Львівська майстерня шоколаду / Lvivska maisternia shokoladu": "Lvivska maisternia shokoladu",
    "Бургери LA П'ЄЦ / Burgers LA PIEC": "Burgers LA PIEC",
    "Дерун / Derun": "Derun",
    "Royal Kebab / Роял Кебаб": "Royal Kebab",
    "Орігамі / Origami": "Origami",
    "Маримо / Marymo": "Marymo",
    "Сніданки LA П'ЄЦ / Breakfast LA PIEC": "Breakfast LA PIEC",
    "Суші Штат / Sushi Shtat": "Sushi Shtat",
    "Сімейна Піца/Сімейна Пекарня": "Сімейна Піца-Сімейна Пекарня",
    "Рамен Мо / Ramen Mo": "Ramen Mo",
}

df["restaurant_name_clean"] = df["restaurant_name"].replace(name_replacements)

# For remaining names containing "/", keep only the first part
df["restaurant_name_clean"] = (
    df["restaurant_name_clean"]
    .str.split("/")
    .str[0]
    .str.strip()
)

# Remove empty values such as "/"
df = df[
    df["restaurant_name_clean"].notna()
    & (df["restaurant_name_clean"].str.strip() != "")
].copy()

In [45]:
# Review names that contain marketing descriptions
df[
    df["restaurant_name_clean"].str.contains(r"\s[-–]\s", na=False)
]["restaurant_name_clean"].drop_duplicates().sort_values()

,restaurant_name_clean
3543,Organica | Органіка - ресторан живої кухні


In [46]:
# Keep only the brand name before the dash
df["restaurant_name_clean"] = (
    df["restaurant_name_clean"]
    .str.split(r"\s[-–]\s")
    .str[0]
    .str.strip()
)

In [47]:
# Official Ukrainian transliteration standard (KMU 2010)
UA_TRANSLIT = {
    'а': 'a',  'б': 'b',  'в': 'v',  'г': 'h',  'ґ': 'g',
    'д': 'd',  'е': 'e',  'є': 'ye', 'ж': 'zh', 'з': 'z',
    'и': 'y',  'і': 'i',  'ї': 'yi', 'й': 'i',  'к': 'k',
    'л': 'l',  'м': 'm',  'н': 'n',  'о': 'o',  'п': 'p',
    'р': 'r',  'с': 's',  'т': 't',  'у': 'u',  'ф': 'f',
    'х': 'kh', 'ц': 'ts', 'ч': 'ch', 'ш': 'sh', 'щ': 'shch',
    'ь': '',   'ю': 'yu', 'я': 'ya', 'ё': 'yo', 'э': 'e',
    'ъ': '',   'ы': 'y',
}

# Generate uppercase transliteration mappings
UA_TRANSLIT.update({
    k.upper(): v.capitalize()
    for k, v in UA_TRANSLIT.items()
    if k.isalpha()
})

def transliterate_ua(name: str) -> str:
    """
    Transliterate Ukrainian Cyrillic to Latin using KMU 2010 standard.
    Leaves already-Latin text unchanged.
    Handles apostrophe and soft sign correctly.
    """
    if pd.isna(name):
        return name

    name = str(name)
    if not any('\u0400' <= char <= '\u04FF' for char in name):
        return name  # already Latin

    result = []
    for char in name:
        result.append(UA_TRANSLIT.get(char, char))

    return ''.join(result)

In [48]:
# Apply transliteration to restaurant names
df["restaurant_name_clean"] = df["restaurant_name_clean"].apply(transliterate_ua)

In [49]:
# Remove special apostrophe-like Unicode characters
# that frequently appear after transliteration.
df["restaurant_name_clean"] = (
    df["restaurant_name_clean"]
    .str.replace(r"[ʹʼ’']", "", regex=True)
)

In [50]:
# Display a random sample to validate the transformation
print("\nSample after standardization:")
print(
    df["restaurant_name_clean"]
    .sample(15, random_state=42)
    .tolist()
)


Sample after standardization:
['Kraft Sushi', 'Cukor RED', '1708 Pizza di Napoli', 'NOA Asia Special', 'Code', 'Cheese Bakery LEM', 'Coffee lab Volodymyra Velykoho', 'Magic Bowls', 'YOKI', 'OKKO CAFÉ', 'Wabi Sabi', 'Monosushi', 'Na Mangal SMOKED BBQ', 'ChaChito', 'YOKI']


In [51]:
df[
    df["restaurant_name_clean"].str.contains(
        "Ye|Yi|Ts|Kh|Shch",
        na=False
    )
]["restaurant_name_clean"].unique()

array(['Khinkalnya na Kryvii lypi', 'Khaip | Pitsa & Sushi',
       'Khinkalnia Na Fedorova'], dtype=object)

In [52]:
display(
    df[
        ["restaurant_name_original", "restaurant_name_clean"]
    ]
    .drop_duplicates()
    .sample(10, random_state=42)
)

,restaurant_name_original,restaurant_name_clean
459,Nectarine / Нектарин,Nectarine
12702,Tretie mistse / Третє місце,Tretie mistse
7932,Na Mangal SMOKED BBQ / На Мангал SMOKED BBQ,Na Mangal SMOKED BBQ
14414,Сімейна Пекарня,Simeina Pekarnya
10153,Uze tut / Уже тут,Uze tut
15203,Khinkalnia Na Fedorova / Хінкальня на Федорова,Khinkalnia Na Fedorova
12599,BATBUT,BATBUT
5371,Royal Kebab / Роял Кебаб,Royal Kebab
12294,eNka Cafe / Обіди та бургери від еНка кафе,eNka Cafe
12062,Emily Brooklyn Pizza,Emily Brooklyn Pizza


### **4.5. Dish Name Transliteration**


Dish names were also transliterated into Latin characters for easier search, dashboard filtering, and text-based analysis.

The original dish names were preserved in dish_name_original, while the transliterated version was stored in dish_name_translit.

In [53]:
# Apply transliteration to dish names
df["dish_name_original"] = df["dish_name"]

df["dish_name_translit"] = (
    df["dish_name"]
    .apply(transliterate_ua)
    .str.replace(r"[ʹʼ’']", "", regex=True)
)

In [54]:
# Display a random sample to validate the transformation
print("\nSample after standardization:")
print(
    df["dish_name_translit"]
    .sample(15, random_state=42)
    .tolist()
)


Sample after standardization:
['Filadelfiya Zapechenyi Losos', 'Tost z obpalenym lososem', 'Filtr kava 1l', 'Tyakhan Yasai (470h)', 'Solodki syrnyky', 'Chyzkeik Rikota Polunytsya Revin', 'Boul indychka', 'Salat z kurkoyu kebab', 'Rol Chervonyi drakon (330h)', 'Min voda Morshynska n/haz 1,5l', 'Kaliforniya z krevetkoyu ta snizhnym krabom (300/50h)', 'Filadelfiya hrin z krevetkoyu', 'Yalovychi rebra', 'Z vyshneyu ta maskarpone (80h)', 'Salat teplyi z moreproduktamy (300h)']


In [55]:
display(
    df[
        ["dish_name_original", "dish_name_translit"]
    ]
    .sample(10, random_state=42)
)

,dish_name_original,dish_name_translit
8134,Філадельфія Запечений Лосось,Filadelfiya Zapechenyi Losos
11051,Тост з обпаленим лососем,Tost z obpalenym lososem
9642,Фільтр кава 1л,Filtr kava 1l
4018,Тяхан Ясай (470г),Tyakhan Yasai (470h)
6364,Солодкі сирники,Solodki syrnyky
8218,Чизкейк Рікота Полуниця Ревінь,Chyzkeik Rikota Polunytsya Revin
5492,Боул індичка,Boul indychka
5859,Салат з куркою кебаб,Salat z kurkoyu kebab
13235,Рол Червоний дракон (330г),Rol Chervonyi drakon (330h)
6822,"Мін вода Моршинська н/газ 1,5л","Min voda Morshynska n/haz 1,5l"


## **5. Final Dataset Validation**

In [56]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

final_missing_values = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": round(df.isna().mean() * 100, 2)
})

print(final_missing_values)

Rows: 15898
Columns: 25
                          missing_count  missing_pct
restaurant_key                        0         0.00
restaurant_name                       0         0.00
rating                                0         0.00
menu_category                         9         0.06
dish_name                             0         0.00
regular_price                         0         0.00
discount_price                    13507        84.96
final_price                           0         0.00
is_discounted                         0         0.00
discount_pct                      13507        84.96
dish_description                   1422         8.94
menu_source_file                      0         0.00
menu_parser_type                      0         0.00
cuisine_type                          0         0.00
cuisine_source_file                 664         4.18
store_url                           664         4.18
rating_clean                       2391        15.04
is_new_restaurant     

**Missing Values Review**

Most missing values are expected and do not indicate data quality problems.

- **discount_price** and **discount_pct** contain a high share of missing values because the majority of menu items were not discounted at the time of data collection.

- **dish_description** is missing for a small subset of products because some restaurants do not provide item descriptions on their Glovo pages.

- **rating_clean** contains missing values for restaurants marked as "Новинка" or restaurants without a displayed rating ("--"). These values were intentionally preserved as nulls because a numerical rating could not be reliably assigned.

- **menu_category** contains only 9 missing values (0.06%), which were successfully handled during category standardization.

- **cuisine_source_file** and **store_url** remain missing for manually completed cuisine records because this information was not available in the original cuisine-category pages.

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15898 entries, 0 to 16120
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   restaurant_key            15898 non-null  object 
 1   restaurant_name           15898 non-null  object 
 2   rating                    15898 non-null  object 
 3   menu_category             15889 non-null  object 
 4   dish_name                 15898 non-null  object 
 5   regular_price             15898 non-null  float64
 6   discount_price            2391 non-null   float64
 7   final_price               15898 non-null  float64
 8   is_discounted             15898 non-null  bool   
 9   discount_pct              2391 non-null   float64
 10  dish_description          14476 non-null  object 
 11  menu_source_file          15898 non-null  object 
 12  menu_parser_type          15898 non-null  object 
 13  cuisine_type              15898 non-null  object 
 14  cuisine_sou

**Final Data Types Review**

The final dataset contains 25 columns and 15,898 rows.

Most columns are stored as text fields because they describe restaurant metadata, menu categories, cuisine labels, source files, URLs, and product descriptions.

Numerical columns include product prices, discount values, discount percentages, and cleaned restaurant ratings.

Boolean columns are used for discount flags and new restaurant identification.

The data types are appropriate for the next analysis stage.

**Newly Created Features**

Several analytical features were created during the preparation process:

| Column | Description |
|----------|----------|
| rating_clean | Numerical restaurant rating extracted from the original rating field |
| is_new_restaurant | Flag identifying restaurants marked as "Новинка" |
| menu_category_clean | Cleaned menu category after text standardization |
| menu_category_group | Final analytical menu category group |
| cuisine_type_en | English translation of cuisine categories |
| restaurant_name_original | Original restaurant name before standardization |
| restaurant_name_clean | Standardized restaurant name used for analysis |
| dish_name_original | Original dish name |
| dish_name_translit | Transliterated dish name using KMU 2010 standard |

In [58]:
summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Restaurant names",
        "Restaurant locations",
        "Unique dishes",
        "Menu category groups",
        "Cuisine type combinations"
    ],
    "value": [
        len(df),
        df["restaurant_name_original"].nunique(),
        df["restaurant_key"].nunique(),
        df["dish_name_original"].nunique(),
        df["menu_category_group"].nunique(),
        df["cuisine_type_en"].nunique()
    ]
})

print(summary)

                      metric  value
0                       Rows  15898
1           Restaurant names    226
2       Restaurant locations    234
3              Unique dishes  11972
4       Menu category groups     37
5  Cuisine type combinations     89


**Final Dataset Summary**

The final analytical dataset contains 15,898 menu items collected from 226 restaurant brands across 234 Glovo store locations in Lviv.

Data quality issues identified during the validation stage were addressed through:
- duplicate removal;
- rating standardization;
- menu category cleaning and consolidation;
- cuisine type completion and translation;
- restaurant name standardization;
- dish name transliteration.

The final dataset includes 37 analytical menu category groups and 89 cuisine type combinations, providing a consistent foundation for pricing, assortment, discount, and restaurant positioning analysis.

The final dataset is ready for further exploratory and business analysis.

It includes cleaned rating fields, standardized menu categories, translated cuisine labels, preserved original restaurant and dish names, and additional standardized text fields for easier analysis and dashboarding.

## **6. Export Clean Dataset**

In [59]:
# Select final analysis-ready columns

final_columns = [
    # Restaurant information
    "restaurant_key",
    "restaurant_name_clean",

    # Restaurant attributes
    "rating_clean",
    "is_new_restaurant",
    "cuisine_type",
    "cuisine_type_en",

    # Menu categories
    "menu_category_clean",
    "menu_category_group",

    # Dish information
    "dish_name_original",
    "dish_name_translit",
    "dish_description",

    # Pricing
    "regular_price",
    "discount_price",
    "final_price",
    "is_discounted",
    "discount_pct"
]

df_final = df[final_columns].copy()

In [60]:
# Validate final export dataset

print("Final export shape:", df_final.shape)

Final export shape: (15898, 16)


In [61]:
# Export cleaned analytical dataset

df_final.to_csv(
    "glovo_menu_cleaned.csv",
    index=False
)

The final analytical dataset contains only the fields required for exploratory analysis, statistical testing, and dashboard development.

Technical parsing columns, source metadata, and intermediate transformation fields were excluded from the export to keep the dataset compact and analysis-ready.

The final dataset includes:

- restaurant identifiers and standardized restaurant names;
- cleaned restaurant ratings and cuisine classifications;
- standardized menu category groups;
- original and transliterated dish names;
- pricing and discount information.

The exported dataset will be used as the primary source for exploratory data analysis and business insights generation.